In [1]:
# if pandas is not yet installed, open the Python command prompt (in your CPLEX environment) and run
#   pip install pandas

import pandas as pd
#df_latlong = pd.read_csv("latlong.csv").set_index('County Name')
df_demand = pd.read_csv("demand.csv").set_index('County')
df_dist = pd.read_csv("distance_km.csv")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
Counties = df_demand.index

In [3]:
Demand = {}
for i in df_demand.index:
    Demand[i] = df_demand['Demand'][i]

In [4]:
CanServe = {}
for i in Counties:
    CanServe[i,i] = 1

for i in df_dist.index:
    if df_dist['Distance'][i] <= 150:
        c1 = df_dist['County1'][i]
        c2 = df_dist['County2'][i]
        CanServe[c1,c2] = 1
        if not c1 in Counties:
            print("ERROR: "+c1+" not in Counties")
        if not c2 in Counties:
            print("ERROR: "+c2+" not in Counties")

In [5]:
baselocation = Demand.keys()
destination = Demand.keys()

In [6]:
from docplex.mp.model import Model
mdl = Model()

In [7]:
# variables
Selectbase = mdl.binary_var_dict(Counties, lb=0, ub=1, name='select base')
Num_Delivery = mdl.continuous_var_dict(CanServe.keys(), lb=0, name='Number of Deliveries')

In [8]:
# objective
mdl.maximize(mdl.sum(Num_Delivery[i,j] for (i,j) in CanServe))

In [9]:
# Constraints: establish 10 drones in Pennsylvania

mdl.add_constraint(mdl.sum(Selectbase[i] for i in Counties) == 10)

docplex.mp.LinearConstraint[](select base_Adams+select base_Allegheny+select base_Armstrong+select base_Beaver+select base_Bedford+select base_Berks+select base_Blair+select base_Bradford+select base_Bucks+select base_Butler+select base_Cambria+select base_Cameron+select base_Carbon+select base_Centre+select base_Chester+select base_Clarion+select base_Clearfield+select base_Clinton+select base_Columbia+select base_Crawford+select base_Cumberland+select base_Dauphin+select base_Delaware+select base_Elk+select base_Erie+select base_Fayette+select base_Forest+select base_Franklin+select base_Fulton+select base_Greene+select base_Huntingdon+select base_Indiana+select base_Jefferson+select base_Juniata+select base_Lackawanna+select base_Lancaster+select base_Lawrence+select base_Lebanon+select base_Lehigh+select base_Luzerne+select base_Lycoming+select base_McKean+select base_Mercer+select base_Mifflin+select base_Monroe+select base_Montgomery+select base_Montour+select base_Northampton+se

In [10]:
# Constraints: Each drone base can handle at most 10000 deliveries per month

for i in baselocation:
    add=mdl.add_constraint(mdl.sum(Num_Delivery[i,j] for j in destination if (i,j) in CanServe.keys()) <= 10000*Selectbase[i])


In [11]:
# Demand Constraints at each destination 

for j in destination:
    mdl.add_constraint(mdl.sum(Num_Delivery[i,j] for i in baselocation if (i,j) in CanServe.keys()) <= Demand[j])
    
    

In [12]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.07301,status='integer optimal solution')

In [13]:
mdl.print_solution()

objective: 100000.000
status: OPTIMAL_SOLUTION(2)
  "select base_Butler"=1
  "select base_Columbia"=1
  "select base_Dauphin"=1
  "select base_Luzerne"=1
  "select base_Schuylkill"=1
  "select base_Snyder"=1
  "select base_Venango"=1
  "select base_Westmoreland"=1
  "select base_Wyoming"=1
  "select base_York"=1
  "Number of Deliveries_Butler_Butler"=1533.000
  "Number of Deliveries_Columbia_Columbia"=561.000
  "Number of Deliveries_Luzerne_Luzerne"=1210.000
  "Number of Deliveries_Venango_Cambria"=1198.000
  "Number of Deliveries_Venango_Warren"=349.000
  "Number of Deliveries_Venango_Allegheny"=195.000
  "Number of Deliveries_Venango_Jefferson"=377.000
  "Number of Deliveries_Venango_Westmoreland"=3044.000
  "Number of Deliveries_Venango_Beaver"=1422.000
  "Number of Deliveries_Venango_Clearfield"=425.000
  "Number of Deliveries_Venango_Indiana"=741.000
  "Number of Deliveries_Venango_Forest"=65.000
  "Number of Deliveries_Venango_Elk"=267.000
  "Number of Deliveries_Venango_Lawrence